# 05 — Final Held-Out Test Evaluation
**PentAI | Goats and Cattle | Google Colab**

Evaluate **one preselected model per species** on its Notebook 02 held-out test split. Produce confusion matrices, classification reports, per-class precision/recall/F1, and documented limitations. **No hyperparameter adjustment or re-selection based on the test set**; if you later retrain after seeing test results, explicitly report that the original held-out test is no longer an untouched benchmark.

## 1. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Local Python session detected. Edit PROJECT_ROOT in the setup cell to your local dataset folder.')

## 2. Configuration and reusable data utilities
Change `RUN_OWNER` in the first cell. For local Jupyter, change `PROJECT_ROOT` and `OUTPUT_ROOT`.

In [ ]:
from pathlib import Path
import os, json, zipfile, shutil, hashlib, random
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
PROJECT_ROOT = Path('/content/drive/MyDrive/Capstone Project')
DATASETS_DIR = PROJECT_ROOT / 'Datasets'
SPLIT_DIR = PROJECT_ROOT / 'outputs' / 'prepared'
WORK_DIR = Path('/content/pentai_data')
# Every member must set their own name: output is private to their MyDrive by default.
RUN_OWNER = 'CHANGE_TO_YOUR_NAME'
if RUN_OWNER == 'CHANGE_TO_YOUR_NAME':
    raise ValueError('Set RUN_OWNER to your own name before running. Do not write to shared team outputs.')
OUTPUT_ROOT = Path('/content/drive/MyDrive/PentAI_Individual_Outputs') / RUN_OWNER
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_NAMES = {'cattle': ['Cow data.zip'],
             'goat': ['Goat Pox Detection.v1i.tensorflow.zip', 'Goat Pox Detection.v1i.tensorflow(1).zip']}
IMG_SIZE=(224,224)
BATCH_SIZE=32
AUTOTUNE=tf.data.AUTOTUNE
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Keep the original ZIP files unchanged. Extract copies only into temporary Colab storage.
def safe_extract_zip(zip_path, target):
    target.mkdir(parents=True, exist_ok=True)
    done = target / '.extraction_complete'
    if done.exists(): return target
    with zipfile.ZipFile(zip_path) as z:
        base = target.resolve()
        for member in z.infolist():
            dest = (target / member.filename).resolve()
            if not dest.is_relative_to(base):
                raise ValueError(f'Unsafe filename in archive: {member.filename}')
        z.extractall(target)
    done.touch()
    return target

def ensure_images(animal):
    target = WORK_DIR / animal
    if (target / '.extraction_complete').exists(): return target
    zips = [DATASETS_DIR / x for x in ZIP_NAMES[animal]]
    found = next((z for z in zips if z.exists()), None)
    if found is None:
        raise FileNotFoundError(f'{animal}: expected one of {zips}. Change ZIP_NAMES if needed.')
    print('Extracting:', found.name)
    return safe_extract_zip(found, target)

def make_path_index(root):
    # Robust against ZIPs that introduce an extra top-level directory.
    paths = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.gif'}]
    by_tail = {}
    for p in paths:
        parts = p.relative_to(root).as_posix().split('/')
        for i in range(len(parts)):
            key = '/'.join(parts[i:]).casefold()
            by_tail.setdefault(key, set()).add(str(p))
    return by_tail

def resolve_manifest_paths(df, root):
    index = make_path_index(root)
    resolved = []
    for relative in df['relative_path'].astype(str):
        key = relative.replace('\\','/').lstrip('/').casefold()
        matches = index.get(key, set())
        if len(matches) != 1:
            raise ValueError(f'Image path must resolve uniquely ({len(matches)} matches): {relative}')
        resolved.append(next(iter(matches)))
    result = df.copy()
    result['absolute_path'] = resolved
    return result

def load_splits(animal, partitions=('train','valid')):
    items=[]
    for part in partitions:
        csv_path=SPLIT_DIR / f'{animal}_{part}.csv'
        if not csv_path.exists(): raise FileNotFoundError(f'Missing Notebook 02 output: {csv_path}')
        frame=pd.read_csv(csv_path, keep_default_na=False)
        if not {'relative_path','class','split'}.issubset(frame.columns):
            raise ValueError(f'{csv_path.name} lacks a required column')
        if frame.empty or not frame['split'].eq(part).all():
            raise ValueError(f'{csv_path.name} is empty or contains the wrong partition')
        if not frame['relative_path'].is_unique: raise ValueError(f'Duplicate paths in {csv_path.name}')
        frame['class'] = frame['class'].astype(str)
        items.append(frame)
    combined=pd.concat(items,ignore_index=True)
    if combined['relative_path'].duplicated().any(): raise ValueError('Image appears in more than one split')
    if 'sha256' in combined and (combined.groupby('sha256')['split'].nunique()>1).any():
        raise ValueError('Exact hash leakage found between splits')
    if 'similarity_group' in combined and (combined.groupby('similarity_group')['split'].nunique()>1).any():
        raise ValueError('Possible similar-image leakage found between splits')
    return resolve_manifest_paths(combined,ensure_images(animal))

def check_approval():
    # The team must first approve the Notebook 01/02 reviews; never infer approval from CSV existence.
    if not TEAM_APPROVED_SPLITS:
        raise RuntimeError('Notebook 02 splits still require team approval. Set TEAM_APPROVED_SPLITS=True ONLY after sign-off.')

def make_ds(df, classes, training=False):
    labels = {label:i for i,label in enumerate(classes)}
    unknown = set(df['class']) - set(labels)
    if unknown: raise ValueError(f'Unexpected labels: {sorted(unknown)}')
    p = df['absolute_path'].astype(str).to_numpy()
    y = np.array([labels[x] for x in df['class']],dtype=np.int32)
    ds=tf.data.Dataset.from_tensor_slices((p,y))
    def read_image(path,label):
        im = tf.io.decode_image(tf.io.read_file(path),channels=3,expand_animations=False)
        im.set_shape((None,None,3))
        im=tf.image.resize(tf.cast(im,tf.float32),IMG_SIZE)
        return im,label
    if training: ds=ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
    return ds.map(read_image,num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

def save_metadata(path, meta):
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(meta,indent=2),encoding='utf-8')

AUG = "tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'),tf.keras.layers.RandomRotation(0.08),tf.keras.layers.RandomZoom(0.10)],name='train_only_augmentation')"

## 3. Confirm approval and frozen model selection

In [ ]:
TEAM_APPROVED_SPLITS = False  # Set only after team approval
check_approval()
selection_path=OUTPUT_ROOT/'models'/'selection.json'
if not selection_path.exists(): raise FileNotFoundError('Run Notebook 04 model selection first')
selection=json.loads(selection_path.read_text())
print({a:x['variant'] for a,x in selection.items()})

## 4. Evaluate held-out test images

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score
import matplotlib.pyplot as plt

all_metrics=[]
for animal in ('cattle','goat'):
    if animal not in selection: raise ValueError(f'No frozen selection for {animal}')
    sel=selection[animal]
    meta=json.loads(Path(sel['metadata_path']).read_text())
    classes=meta['classes']
    frame=load_splits(animal,('test',))
    if set(frame['class'])-set(classes): raise ValueError(f'{animal}: unseen test label')
    ds=make_ds(frame,classes)
    model=tf.keras.models.load_model(sel['model_path'])
    proba=model.predict(ds,verbose=0)
    pred=np.argmax(proba,axis=1)
    expected=np.array([classes.index(label) for label in frame['class']])
    report=classification_report(expected,pred,labels=list(range(len(classes))),target_names=classes,
                                 output_dict=True,zero_division=0)
    cm=confusion_matrix(expected,pred,labels=list(range(len(classes))))
    eval_dir=OUTPUT_ROOT/'evaluation'/animal
    eval_dir.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(report).T.to_csv(eval_dir/'classification_report.csv')
    pd.DataFrame(cm,index=classes,columns=classes).to_csv(eval_dir/'confusion_matrix.csv')
    predictions=frame[['relative_path','class']].copy()
    predictions['predicted_class']=[classes[i] for i in pred]
    predictions['confidence']=proba.max(axis=1)
    predictions['correct']=predictions['class'].eq(predictions['predicted_class'])
    predictions.to_csv(eval_dir/'test_predictions.csv',index=False)
    fig,ax=plt.subplots(figsize=(max(5,len(classes)*1.1),max(4,len(classes)*.9)))
    im=ax.imshow(cm,cmap='Blues')
    ax.set(xticks=range(len(classes)),yticks=range(len(classes)),xticklabels=classes,yticklabels=classes,
           xlabel='Predicted',ylabel='Actual',title=f'{animal} held-out test confusion matrix')
    plt.setp(ax.get_xticklabels(),rotation=45,ha='right')
    for (i,j),v in np.ndenumerate(cm): ax.text(j,i,str(v),ha='center',va='center')
    fig.colorbar(im,ax=ax); fig.tight_layout();fig.savefig(eval_dir/'confusion_matrix.png',dpi=140);plt.show()
    summary={'animal':animal,'selected_variant':sel['variant'],'classes':classes,
             'test_samples':len(frame),'accuracy':float(accuracy_score(expected,pred)),
             'balanced_accuracy':float(balanced_accuracy_score(expected,pred)),
             'macro_f1':float(report['macro avg']['f1-score']),
             'weighted_f1':float(report['weighted avg']['f1-score']),
             'note':'Single held-out evaluation; not a clinical or veterinary diagnostic validation'}
    save_metadata(eval_dir/'metrics.json',summary)
    all_metrics.append(summary)
    print(animal,summary)
pd.DataFrame(all_metrics).to_csv(OUTPUT_ROOT/'evaluation'/'final_metrics.csv',index=False)

## 5. Limitations and interpretation
Review mistakes against the original images and annotator records. Discuss class imbalance, dataset shift, small evaluation sample sizes, possible photographic shortcuts, and uncertain labels. **No model here should be presented as a validated veterinary diagnostic tool.**

In [ ]:
print('Generated for each species: classification_report.csv, confusion_matrix.csv/png, test_predictions.csv, metrics.json')
print('Overall comparison:',OUTPUT_ROOT/'evaluation'/'final_metrics.csv')